In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, matthews_corrcoef
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# Налаштування візуалізації
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

1. Підготовка та обробка даних

In [5]:
# 1. Завантаження датасету
df = pd.read_csv('global_coffee_health.csv')

# 2. Видалення неінформативної колонки ID
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

# 3. Кодування цільової змінної (Health_Issues)
# Перетворення текстових міток (None, Mild, Moderate, Severe) у числа 0, 1, 2, 3
le = LabelEncoder()
df['Health_Issues'] = le.fit_transform(df['Health_Issues'])
target_classes = le.classes_
print(f"Класи цільової змінної: {target_classes}")

# 4. Кодування категоріальних ознак (One-Hot Encoding)
# Автоматично знаходить колонки типу 'object' та перетворює їх у dummy-змінні
df_encoded = pd.get_dummies(df, drop_first=True)

# 5. Формування матриці ознак X та вектора цільової змінної y
X = df_encoded.drop('Health_Issues', axis=1).values
y = df_encoded['Health_Issues'].values
feature_names = df_encoded.drop('Health_Issues', axis=1).columns.tolist()

# 6. Поділ на тренувальну та тестову вибірки (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("-" * 30)
print(f"Розмірність X_train: {X_train.shape}")
print(f"Розмірність X_test:  {X_test.shape}")

Класи цільової змінної: ['Mild' 'Moderate' 'Severe' nan]
------------------------------
Розмірність X_train: (8000, 39)
Розмірність X_test:  (2000, 39)


2. Реалізація власного класу Дерева Рішень (MyDecisionTree)

In [ ]:
class MyDecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        self.feature_importances_ = None
        self.n_total = 0

    # Обчислення Gini impurity [cite: 11-12]
    def _gini(self, y):
        m = len(y)
        if m == 0:
            return 0
        probs = [np.sum(y == c) / m for c in np.unique(y)]
        return 1 - sum(p ** 2 for p in probs)

    # Пошук найкращого розбиття [cite: 31-32]
    def _best_split(self, X, y):
        m, n = X.shape
        if m <= self.min_samples_split:
            return None, None, None

        parent_gini = self._gini(y)
        best_gini = float('inf')
        best_feature, best_threshold = None, None

        # Перебір всіх ознак
        for feature_idx in range(n):
            thresholds = np.unique(X[:, feature_idx])
            
            # Для оптимізації, якщо унікальних значень забагато, можна брати не всі
            if len(thresholds) > 100:
                thresholds = np.percentile(thresholds, np.linspace(0, 100, 10))

            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                y_left, y_right = y[left_mask], y[right_mask]
                
                # Зважений Gini [cite: 15-18]
                gini_left = self._gini(y_left)
                gini_right = self._gini(y_right)
                weighted_gini = (len(y_left) / m) * gini_left + (len(y_right) / m) * gini_right

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature_idx
                    best_threshold = threshold

        # Розрахунок gain для feature importance [cite: 24]
        # Gain = (Gini_parent - Gini_weighted) * (N_node / N_total)
        gini_gain = 0
        if best_feature is not None:
            gini_gain = (parent_gini - best_gini) * (m / self.n_total)
            
        return best_feature, best_threshold, gini_gain

    # Рекурсивна побудова дерева [cite: 34, 54]
    def _build_tree(self, X, y, depth=0):
        # Перевірка умов зупинки
        n_samples, _ = X.shape
        n_labels = len(np.unique(y))

        if depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split:
            leaf_value = np.bincount(y).argmax() if len(y) > 0 else 0
            return {'leaf': True, 'class': leaf_value}

        feature_idx, threshold, gain = self._best_split(X, y)

        if feature_idx is None:
            leaf_value = np.bincount(y).argmax()
            return {'leaf': True, 'class': leaf_value}

        # Оновлення важливості ознак [cite: 32]
        self.feature_importances_[feature_idx] += gain

        left_mask = X[:, feature_idx] <= threshold
        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)

        return {
            'leaf': False,
            'feature': feature_idx,
            'threshold': threshold,
            'left': left_subtree,
            'right': right_subtree
        }

    # Навчання моделі [cite: 59-68]
    def fit(self, X, y):
        self.n_total = len(y)
        self.feature_importances_ = np.zeros(X.shape[1])
        self.tree = self._build_tree(X, y)
        
        # Нормалізація важливості [cite: 25-27]
        total_importance = np.sum(self.feature_importances_)
        if total_importance > 0:
            self.feature_importances_ /= total_importance
            
        return self

    # Прогноз для одного зразка [cite: 69]
    def _predict_one(self, x, node):
        if node['leaf']:
            return node['class']
        if x[node['feature']] <= node['threshold']:
            return self._predict_one(x, node['left'])
        else:
            return self._predict_one(x, node['right'])

    # Прогноз для всіх зразків [cite: 71]
    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

print("Клас MyDecisionTree реалізовано.")